# Cross-protein sequon retention — CARBonAra

Occupied sequons against motif-only sequons **in the same chain**. Both classes
sit on one backbone and share one set of 32 unconstrained designs, so the
contrast is within-protein by construction — unlike the matched-secretory
comparison, which pairs across proteins and cannot fully control protein
identity.

This notebook does one thing: generate designs over
`candidate_manifest_dataset.csv` with `--save-sequences`, then run
`57_cross_protein_sequon_retention.py` on them. ProteinMPNN has already been run
on a laptop; this covers a model that cannot be installed there.

**Cost:** one-shot per position, so faster than the autoregressive models.

Sibling notebooks cover the other models. They are separate because `fair-esm`
(ESM-IF) and EvolutionaryScale's `esm` (ESM3) both install a top-level module
called `esm` and cannot share a runtime.

## 1. GPU

In [ ]:
import subprocess

# FileNotFoundError, not empty output, is what a runtime without the driver
# gives -- so catching it is the whole point of this cell.
try:
    listing = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
except FileNotFoundError:
    listing = ''
print(listing.strip() or
      'NO GPU -- Runtime > Change runtime type > T4 GPU, then rerun from here.')

## 2. Dependencies

In [ ]:
import subprocess, sys

def pip(*a):
    print("$ pip", *a, flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

# Upstream's src/__init__.py imports its whole src package, so the scoring and
# dataset modules' dependencies are needed even though this integration calls
# none of them. Same list as venv-carbonara in scripts/arc/glyco_setup.sh.
pip("gemmi", "blosum", "scikit-learn", "h5py", "tqdm")
pip("pandas", "scipy", "biopython")
print("dependencies ready")

In [ ]:
import os, shutil, subprocess
from pathlib import Path

CARBONARA_MODEL = 's_v6_4_2022-09-16_11-51'
URL  = 'https://github.com/LBM-EPFL/CARBonAra.git'
ROOT = Path('/content/CARBonAra')
# What the integration actually needs. `runner_support.carbonara_dir()`
# identifies the checkout by carbonara.py, and carbonara.py itself falls back to
# `from src...`, so a checkout missing either is useless however many weights it
# has.
ENTRY, SRC = ROOT / 'carbonara.py', ROOT / 'src'
WEIGHTS = ROOT / f'model/save/{CARBONARA_MODEL}/model.pt'


def complete():
    return (ENTRY.is_file() and SRC.is_dir()
            and WEIGHTS.is_file() and WEIGHTS.stat().st_size > 1e6)


# Identified by what it must contain, not by the directory existing: an
# interrupted clone leaves a tree that looks finished and is not, and an
# existence check would then skip repairing it forever.
if not complete():
    shutil.rmtree(ROOT, ignore_errors=True)
    # The repository is 1.1 GB, of which 838 MB is the authors' own result
    # files. Sparse-checkout takes the entry point, src/ and one checkpoint.
    sparse = subprocess.run(
        ['git', 'clone', '--depth', '1', '--filter=blob:none', '--sparse', '-q',
         URL, str(ROOT)]).returncode == 0
    if sparse:
        subprocess.run(['git', '-C', str(ROOT), 'sparse-checkout', 'set',
                        'src', f'model/save/{CARBONARA_MODEL}'], check=True)
        # Cone mode is supposed to keep root-level files, and does on git 2.50 --
        # but it did not on Colab, and carbonara.py is exactly the file the
        # checkout is identified by. Restore the root explicitly rather than
        # trusting the mode; with a blob:none clone this fetches just these.
        if not ENTRY.is_file():
            print('root files absent from the sparse checkout; restoring them',
                  flush=True)
            subprocess.run(['git', '-C', str(ROOT), 'checkout', 'HEAD', '--',
                            'carbonara.py', '__init__.py'])
    if not complete():
        print('sparse checkout incomplete; falling back to a full clone (1.1 GB)',
              flush=True)
        shutil.rmtree(ROOT, ignore_errors=True)
        subprocess.run(['git', 'clone', '--depth', '1', '-q', URL, str(ROOT)],
                       check=True)

os.environ['CARBONARA_DIR'] = str(ROOT)
assert complete(), (
    f'CARBonAra checkout unusable: carbonara.py={ENTRY.is_file()} '
    f'src={SRC.is_dir()} weights={WEIGHTS.is_file()}. --model carbonara '
    'cannot run. Delete /content/CARBonAra and rerun this cell.')
print(f'CARBonAra ready: carbonara.py, src/, weights '
      f'{WEIGHTS.stat().st_size / 1e6:.1f} MB')

## 3. The code

Fetch and reset rather than skipping when the directory exists. Restarting a
Colab session restarts the kernel but keeps `/content`, so a checkout from an
earlier run survives — and a clone guarded only by directory existence then
silently keeps stale code.

In [ ]:
import os, subprocess, sys
from pathlib import Path

BRANCH = 'fix/context-extractor-mapping'
REPO   = 'https://github.com/LBDillon/Glycan-occupancy-analysis.git'
MODULE = '/content/module'

if os.path.exists(MODULE):
    subprocess.run(['git', '-C', MODULE, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', MODULE, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO, MODULE], check=True)

head = subprocess.run(['git', '-C', MODULE, 'log', '--oneline', '-1'],
                      capture_output=True, text=True).stdout.strip()
print('HEAD:', head)

# This notebook depends on two things that are newer than most of the branch.
# Check they arrived rather than discovering it after the bundle download.
analysis = Path(MODULE) / 'glyco_context/pipeline/57_cross_protein_sequon_retention.py'
support  = Path(MODULE) / 'src/experimental_glycosylation_sites/runner_support.py'
assert analysis.exists(), (
    f'{analysis.name} absent. The branch may not be pushed, or this is a stale '
    'clone: rm -rf /content/module and rerun this cell.')
assert '--save-sequences' in support.read_text(), (
    'this checkout of 08_design.py cannot save sequences, so the analysis below '
    'has nothing to read. Pull the branch again.')
print('save-sequences and stage 57 present')

os.chdir(MODULE)
sys.path.insert(0, f'{MODULE}/src')
sys.path.insert(0, f'{MODULE}/glyco_context/src')

## 4. Structures and the manifest

The release bundle carries every structure the dataset manifest names, plus the
manifests themselves — `results/` is gitignored, so the clone has neither.

In [ ]:
import shutil, subprocess, tarfile, gzip, time
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

RELEASE_TAG = 'bundle-2026-08-20'
REPO_SLUG   = 'LBDillon/Glycan-occupancy-analysis'
release_url = f'https://github.com/{REPO_SLUG}/releases/download/{RELEASE_TAG}/colab_bundle.tar'

drive_tar = Path('/content/drive/MyDrive/sugarfix/colab_bundle.tar')
local_tar = Path('/content/bundle.tar')

def _ok(path, min_gb=0.4):
    return path.exists() and path.stat().st_size / 1e9 >= min_gb

if _ok(drive_tar):
    BUNDLE_TAR = drive_tar
    print(f'using Drive copy: {BUNDLE_TAR}')
else:
    if not _ok(local_tar):
        print(f'downloading {release_url}', flush=True)
        subprocess.run(['wget', '-q', '--show-progress', '-O', str(local_tar), release_url])
    # A 404 writes a small HTML page rather than failing, so check the size --
    # otherwise the error surfaces later as an unreadable tar.
    assert _ok(local_tar), f'download failed or returned an error page: {local_tar}'
    BUNDLE_TAR = local_tar

work = Path('/content/bundle')
need = work / 'structures'
# Guard on extracted CONTENT, not on the directory existing: mkdir runs before
# extractall, so a tar that is missing leaves an empty directory behind and an
# existence check would skip the extraction forever.
if not (need.is_dir() and any(need.iterdir())):
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True, exist_ok=True)
    print(f'extracting {BUNDLE_TAR} ...', flush=True)
    with tarfile.open(BUNDLE_TAR) as tar:
        tar.extractall(work, filter='data')
if not need.is_dir():
    nested = [d for d in work.iterdir() if d.is_dir() and (d / 'structures').is_dir()]
    if nested:
        work, need = nested[0], nested[0] / 'structures'
assert need.is_dir(), f'no structures/ under {work}: {[p.name for p in work.iterdir()]}'

STRUCT = Path(MODULE) / 'data/cache/pdb'
STRUCT.mkdir(parents=True, exist_ok=True)
gz    = list(need.glob('*.gz'))
plain = [p for p in need.iterdir() if p.suffix in ('.pdb', '.cif')]
t0 = time.time()
for i, p in enumerate(gz, 1):
    target = STRUCT / p.stem
    if not target.exists():
        with gzip.open(p, 'rb') as fh, open(target, 'wb') as out:
            shutil.copyfileobj(fh, out)
    if i % 400 == 0:
        print(f'  {i}/{len(gz)} ({time.time()-t0:.0f}s)', flush=True)
for p in plain:
    if not (STRUCT / p.name).exists():
        shutil.copy(p, STRUCT / p.name)

for sub in ('manifests', 'matching'):
    src, dst = work / sub, Path(MODULE) / 'results' / sub
    dst.mkdir(parents=True, exist_ok=True)
    for p in src.glob('*.csv'):
        shutil.copy(p, dst / p.name)

MANIFEST = Path(MODULE) / 'results/manifests/candidate_manifest_dataset.csv'
assert MANIFEST.exists(), f'{MANIFEST} not in the bundle'
import pandas as pd
_m = pd.read_csv(MANIFEST, low_memory=False)
_m = _m[_m.scoreable.astype(bool)] if 'scoreable' in _m else _m
print(f"\nstructures: {len(list(STRUCT.iterdir()))}")
print(f"manifest:   {len(_m)} scoreable sites in "
      f"{_m.groupby(['structure_pdb_id','structure_chain_id']).ngroups} chains")

## 5. Preflight

Two designs on one chain, before spending the GPU hours. This catches the
failure that otherwise looks like success: a run that resolves no structures
writes an empty table and exits zero.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from experimental_glycosylation_sites.runner_support import structure_paths, build_adapter
from experimental_glycosylation_sites.structures import _parse_chains

paths = structure_paths(())
print(f'structures resolvable: {len(paths)}')
assert len(paths) > 100, 'almost nothing resolved -- the bundle step did not land'

groups = list(_m.groupby(['structure_pdb_id', 'structure_chain_id']))
missing = [k for k, _ in groups if str(k[0]).upper() not in paths]
print(f'chains without a structure: {len(missing)}/{len(groups)}')

(pdb, chain), _grp = next((k, g) for k, g in groups if str(k[0]).upper() in paths)
adapter = build_adapter('carbonara', 'cuda', max_batch=None)
print('provenance:', adapter.describe())

path = paths[str(pdb).upper()]
native = next(c for c in _parse_chains(path, str(pdb)) if c.chain_id == str(chain))
designs = adapter.design(path, chain, n_designs=2, temperature=0.1, seed=0)

# Full length, or every index in the retention read-out is shifted.
assert all(len(d) == len(native.sequence) for d in designs), 'a design changed length'
assert len(set(designs)) > 1, 'every design identical -- temperature or seed is wrong'
identity = sum(a == b for a, b in zip(designs[0], native.sequence)) / len(native.sequence)
print(f'{pdb}:{chain}  L={len(native.sequence)}  identity to native {identity:.1%}')
print('preflight ok')

## 6. The design run

`--save-sequences` keeps the sequences stage 08 otherwise discards after
classifying the manifest sites. That is the whole point: every position-level
question afterwards is offline analysis rather than another GPU run.

**A fresh output path matters.** Resume keys on manifest sites, so resuming onto
an existing table skips chains whose sites are already recorded and would leave
their sequences unwritten. The stage warns when this applies.

In [ ]:
import shutil, subprocess, sys, time
from pathlib import Path

DRIVE = Path('/content/drive/MyDrive/glyco_occupancy/cross_protein_carbonara')
DRIVE.mkdir(parents=True, exist_ok=True)

# Stage 08 writes STRAIGHT TO DRIVE, not to /content. It flushes every 25
# chains, so a disconnect costs at most those 25 -- whereas a run writing under
# /content loses everything, because save_to_drive only fires when the run ends
# and the runtime is gone by then. That is survivable for a 40-minute job and
# not for a multi-hour one.
OUT = DRIVE / Path('results/designs/retention_dataset_unconstrained_carbonara.csv').name
SEQ = OUT.with_name(OUT.stem + '_sequences.csv')


def usable(path):
    """Whether a checkpoint holds work stage 08 can actually resume.

    Two schemas share the retention_ prefix: the site table, keyed on accession,
    and the sequence table --save-sequences writes, keyed on chain. Checking
    every retention_ file for an accession column silently refuses to restore
    the sequences, and the run then resumes with the site table complete, every
    chain skipped, and nothing for stage 57 to read.
    """
    if path.stat().st_size == 0:
        return False, 'empty'
    if not path.name.startswith('retention_'):
        return True, ''
    expected = 'sequence' if path.name.endswith('_sequences.csv') else 'accession'
    if expected not in path.open().readline():
        return False, 'headerless'
    return True, ''


# Nothing to restore -- the outputs already live in Drive. What does need doing
# is discarding a checkpoint with no resumable work in it, since stage 08 would
# otherwise refuse the file or, worse, append to a headerless one.
for path in (OUT, SEQ, OUT.with_name(OUT.stem + '_failures.csv')):
    if path.exists():
        ok, why = usable(path)
        if not ok:
            print(f'discarding ({why}): {path.name}'); path.unlink()

if OUT.exists():
    import pandas as _pd
    print(f'resuming: {len(_pd.read_csv(OUT, low_memory=False))} sites already done')
else:
    print('starting fresh')


started = time.time()
done = subprocess.run(
    [sys.executable, '-u', 'pipeline/08_design.py', str(MANIFEST), str(OUT),
     '--model', 'carbonara', '--device', 'cuda', '--save-sequences'],
    capture_output=True, text=True, cwd=MODULE)
print(done.stdout[-6000:] or '(no stdout)')
if done.returncode:
    print('--- STDERR ---'); print(done.stderr[-4000:] or '(no stderr)')
    raise SystemExit(f'stage 08 exited {done.returncode}; the cause is above')
print(f'elapsed {(time.time() - started) / 60:.0f} min')

## 7. Coverage

A rate over an unrecorded subset cannot be checked later, so every site is
accounted for as designed, failed, or neither — and "neither" is the one that
matters.

In [ ]:
import pandas as pd
KEY = ['accession', 'position', 'structure_pdb_id', 'structure_chain_id']

designed = pd.read_csv(OUT, low_memory=False)
fail_path = OUT.with_name(OUT.stem + '_failures.csv')
failures = pd.DataFrame(columns=KEY)
if fail_path.exists() and fail_path.stat().st_size > 1:
    try:
        failures = pd.read_csv(fail_path, low_memory=False)
    except pd.errors.EmptyDataError:
        pass

expected = len(_m.drop_duplicates(KEY))
parts = [designed.reindex(columns=KEY)]
if len(failures):
    parts.append(failures.reindex(columns=KEY))
accounted = len(pd.concat(parts, ignore_index=True).drop_duplicates(KEY))
short = expected - accounted
print(f'{len(designed)} designed + {len(failures)} failed = {accounted} of {expected}'
      + ('' if short <= 0 else f'   <-- {short} NEVER ATTEMPTED'))

seqs = pd.read_csv(SEQ, low_memory=False)
chains_designed = designed.groupby(['structure_pdb_id', 'structure_chain_id']).ngroups
chains_seqs = seqs.groupby(['structure_pdb_id', 'structure_chain_id']).ngroups
print(f'sequences: {len(seqs)} designs over {chains_seqs} chains '
      f'(designed table covers {chains_designed})')
assert chains_seqs >= chains_designed, (
    'fewer chains in the sequence file than in the site table -- this is the '
    'resume trap: rerun into a FRESH --out path so every chain is regenerated.')
if short > 0:
    raise SystemExit('rerun the design cell; it resumes and will fill these in')
print('\nevery site accounted for')

## 8. The analysis

Offline from here: no model is loaded. Occupied against motif-only sequons in
the same chain, plus the exact-triplet control, bootstrapped over proteins.

In [ ]:
done = subprocess.run(
    [sys.executable, '-u', 'glyco_context/pipeline/57_cross_protein_sequon_retention.py',
     '--sequences', str(SEQ), '--label', 'cross_protein_carbonara'],
    capture_output=True, text=True, cwd=MODULE)
print(done.stdout[-6000:] or '(no stdout)')
if done.returncode:
    print('--- STDERR ---'); print(done.stderr[-4000:])
    raise SystemExit(f'stage 57 exited {done.returncode}')

for path in (Path(MODULE) / 'glyco_context/results/analysis').glob('cross_protein_carbonara_*'):
    shutil.copy(path, DRIVE / path.name)
print('\nanalysis copied to Drive')

## 9. Back to the laptop

Standalone: needs no other cell in this session to have run. Lays the files out
the way the repo expects, so the archive unzips into the project root.

In [ ]:
import zipfile
from pathlib import Path
from google.colab import drive, files

try:
    drive.mount('/content/drive')
except Exception as exc:
    print('mount:', str(exc)[:80])

DRIVE = Path('/content/drive/MyDrive/glyco_occupancy/cross_protein_carbonara')
MODULE = Path('/content/module')


def route(name):
    # Any cross_protein_* file is stage 57 output, including the validation
    # arm's, which is named for proteinmpnn rather than for this notebook.
    if name.endswith('.json') or name.startswith('cross_protein_'):
        return 'glyco_context/results/analysis'
    return 'results/designs'


found = {}
if DRIVE.is_dir():
    for path in DRIVE.iterdir():
        if path.is_file():
            found[path.name] = path
for pattern in ('results/designs/retention_dataset_unconstrained_*.csv',
                'glyco_context/results/analysis/cross_protein_*'):
    for path in MODULE.glob(pattern):
        if (path.name not in found
                or path.stat().st_mtime > found[path.name].stat().st_mtime):
            found[path.name] = path

archive = Path('/content/cross_protein_carbonara_results.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    for name, path in sorted(found.items()):
        bundle.write(path, f'{route(name)}/{name}')
print(f'{len(found)} files, {archive.stat().st_size / 1e6:.2f} MB')
for name in sorted(found):
    print('  ', name)
files.download(str(archive))